
## 1. What is Pydantic?

**Pydantic is a Python library for data validation and structured data parsing using Python type hints.**

In simple terms:

> **You define what data should look like, and Pydantic checks whether the actual data follows those rules.**

For example:

```python
from pydantic import BaseModel

class ModelConfig(BaseModel):
    model_name: str
    learning_rate: float
    epochs: int

config = ModelConfig(
    model_name="xgboost",
    learning_rate=0.1,
    epochs=100
)

print(config)

Output:

model_name='xgboost' learning_rate=0.1 epochs=100

If someone gives:

```python
config = ModelConfig(
    model_name="xgboost",
    learning_rate="abc",
    epochs=100
)

Pydantic raises a validation error because `"abc"` cannot be converted to a valid float.


# 2. Why is Pydantic useful in MLOps?

Think about a typical ML production pipeline:

```text
                ┌──────────────┐
                │ Configuration │
                └──────┬───────┘
                       ↓
              ┌─────────────────┐
              │ Data Validation │
              └────────┬────────┘
                       ↓
              ┌─────────────────┐
              │ Feature Pipeline│
              └────────┬────────┘
                       ↓
              ┌─────────────────┐
              │ Model Training  │
              └────────┬────────┘
                       ↓
              ┌─────────────────┐
              │ Model Registry  │
              └────────┬────────┘
                       ↓
              ┌─────────────────┐
              │ Model API       │
              └─────────────────┘
```

Pydantic can be used at almost every boundary.

### Common MLOps use cases

| Use case                 | Pydantic's role                     |
| ------------------------ | ----------------------------------- |
| Configuration            | Validate training configuration     |
| Hyperparameters          | Validate allowed values             |
| API input                | Validate prediction requests        |
| API output               | Validate prediction responses       |
| Data contracts           | Ensure expected fields/types        |
| Environment variables    | Parse configuration                 |
| Pipeline parameters      | Validate pipeline inputs            |
| Model metadata           | Define structured metadata          |
| Experiment configuration | Validate experiment settings        |
| FastAPI                  | Automatically validate API requests |

---

# 3. Pydantic vs normal Python

Without Pydantic:

```python
learning_rate = config["learning_rate"]
epochs = config["epochs"]
model_name = config["model_name"]
```

You have to manually check:

```python
if not isinstance(learning_rate, float):
    raise ValueError("Invalid learning rate")

if not isinstance(epochs, int):
    raise ValueError("Invalid epochs")
```

With Pydantic:

```python
class TrainingConfig(BaseModel):
    learning_rate: float
    epochs: int
    model_name: str
```

Validation happens automatically.

---

# 4. Install Pydantic

For Pydantic V2:

```bash
pip install pydantic
```

Check:

```bash
pip show pydantic
```

If you are using FastAPI:

```bash
pip install fastapi uvicorn pydantic
```

---

# 5. Basic Pydantic model

Create:

```text
mlops_project/
│
├── config.py
├── train.py
└── requirements.txt
```

`config.py`

```python
from pydantic import BaseModel


class TrainingConfig(BaseModel):
    model_name: str
    learning_rate: float
    epochs: int
    batch_size: int
```

Then:

```python
from config import TrainingConfig

config = TrainingConfig(
    model_name="xgboost",
    learning_rate=0.1,
    epochs=100,
    batch_size=32
)

print(config)
```

---

# 6. Validation

Now intentionally provide bad data:

```python
config = TrainingConfig(
    model_name="xgboost",
    learning_rate=-0.1,
    epochs="hello",
    batch_size=32
)
```

Pydantic will detect invalid fields.

But we can make validation much stronger.

---

# 7. Add constraints

For ML, you often want:

```text
learning_rate > 0
epochs > 0
batch_size > 0
```

Pydantic supports this directly.

```python
from pydantic import BaseModel, Field


class TrainingConfig(BaseModel):

    model_name: str

    learning_rate: float = Field(
        gt=0,
        le=1
    )

    epochs: int = Field(
        gt=0,
        le=10000
    )

    batch_size: int = Field(
        gt=0
    )
```

Now:

```python
config = TrainingConfig(
    model_name="xgboost",
    learning_rate=0.1,
    epochs=100,
    batch_size=32
)
```

Valid.

But:

```python
learning_rate=-0.5
```

will fail.

---

# 8. This becomes very useful in MLOps

Imagine your training pipeline receives:

```yaml
model_name: xgboost
learning_rate: 0.1
epochs: 100
batch_size: 32
```

You can load this configuration and validate it before training.

For example:

```python
from pydantic import BaseModel, Field
import yaml


class TrainingConfig(BaseModel):
    model_name: str
    learning_rate: float = Field(gt=0, le=1)
    epochs: int = Field(gt=0)
    batch_size: int = Field(gt=0)


with open("config.yaml") as f:
    data = yaml.safe_load(f)

config = TrainingConfig(**data)

print(config)
```

Your pipeline becomes:

```text
config.yaml
     ↓
YAML parser
     ↓
Pydantic
     ↓
Validation
     ↓
Training
```

If validation fails:

```text
       config.yaml
            ↓
         Pydantic
            ↓
       ❌ Validation
            ↓
      Stop pipeline
```

This is much better than allowing bad parameters to reach model training.

---

# 9. Pydantic for ML hyperparameters

This is one of the most useful applications.

Suppose you have XGBoost:

```python
from pydantic import BaseModel, Field


class XGBoostConfig(BaseModel):

    n_estimators: int = Field(gt=0)

    max_depth: int = Field(
        gt=0,
        le=20
    )

    learning_rate: float = Field(
        gt=0,
        le=1
    )

    subsample: float = Field(
        gt=0,
        le=1
    )

    colsample_bytree: float = Field(
        gt=0,
        le=1
    )
```

Then:

```python
config = XGBoostConfig(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)
```

Now you have a validated hyperparameter object.

---

# 10. Pydantic + Optuna

Since you're learning **Optuna**, this is a very good combination.

Optuna generates hyperparameters:

```python
learning_rate = trial.suggest_float(
    "learning_rate",
    0.001,
    0.3
)

max_depth = trial.suggest_int(
    "max_depth",
    3,
    10
)
```

You can pass those parameters into Pydantic:

```python
params = XGBoostConfig(
    n_estimators=500,
    max_depth=max_depth,
    learning_rate=learning_rate,
    subsample=0.8,
    colsample_bytree=0.8
)
```

Architecture:

```text
             Optuna
                │
                ↓
       Hyperparameters
                │
                ↓
            Pydantic
                │
          Validation
                │
                ↓
          XGBoost Model
                │
                ↓
           Evaluation
```

This gives you a clean **hyperparameter contract**.

---

# 11. Pydantic for ML API

This is probably the most important production use case.

Suppose your model predicts house prices.

API request:

```json
{
    "area": 1500,
    "bedrooms": 3,
    "bathrooms": 2
}
```

Define:

```python
from pydantic import BaseModel, Field


class PredictionRequest(BaseModel):

    area: float = Field(gt=0)

    bedrooms: int = Field(gt=0)

    bathrooms: int = Field(gt=0)
```

Then FastAPI:

```python
from fastapi import FastAPI

app = FastAPI()


@app.post("/predict")
def predict(request: PredictionRequest):

    prediction = model.predict([
        [
            request.area,
            request.bedrooms,
            request.bathrooms
        ]
    ])

    return {
        "prediction": prediction[0]
    }
```

Now the API automatically validates requests.

---

# 12. What happens with bad input?

Suppose somebody sends:

```json
{
    "area": -100,
    "bedrooms": 0,
    "bathrooms": 2
}
```

Pydantic rejects it before your ML model receives the data.

This is extremely important in production.

```text
User
 │
 │ JSON
 ↓
FastAPI
 │
 ↓
Pydantic
 │
 ├── ❌ Invalid → 400/422 response
 │
 └── ✅ Valid
        ↓
      Model
        ↓
    Prediction
```

---

# 13. Pydantic for prediction response

You can also validate what your model returns.

```python
class PredictionResponse(BaseModel):

    prediction: float
    model_version: str
    confidence: float
```

Then:

```python
return PredictionResponse(
    prediction=125000.0,
    model_version="v1.3",
    confidence=0.91
)
```

Now your API has a defined output contract.

---

# 14. Pydantic for model metadata

In MLOps, models usually have metadata.

For example:

```python
class ModelMetadata(BaseModel):

    model_name: str
    model_version: str
    framework: str
    accuracy: float
    training_dataset: str
    git_commit: str
```

Example:

```python
metadata = ModelMetadata(
    model_name="customer_churn",
    model_version="1.2.0",
    framework="XGBoost",
    accuracy=0.91,
    training_dataset="customer_data_v5",
    git_commit="a82f91c"
)
```

This creates a standardized model metadata structure.

---

# 15. Pydantic settings for environment variables

This is particularly useful for **production MLOps**.

For example:

```text
MODEL_NAME=xgboost
MODEL_VERSION=1.2
DATABASE_URL=...
MLFLOW_TRACKING_URI=...
```

You don't want these values hard-coded.

Pydantic settings can load and validate environment configuration.

With the current Pydantic ecosystem, install:

```bash
pip install pydantic-settings
```

Then:

```python
from pydantic_settings import BaseSettings


class Settings(BaseSettings):

    model_name: str
    model_version: str
    mlflow_tracking_uri: str

    class Config:
        env_file = ".env"


settings = Settings()
```

`.env`:

```text
MODEL_NAME=xgboost
MODEL_VERSION=1.2
MLFLOW_TRACKING_URI=http://localhost:5000
```

Then:

```python
print(settings.model_name)
```

---

# 16. A realistic MLOps project

You could structure your project like this:

```text
mlops_project/
│
├── app/
│   ├── main.py
│   ├── schemas.py
│   └── config.py
│
├── training/
│   ├── train.py
│   ├── model.py
│   └── hyperparameters.py
│
├── data/
│
├── models/
│
├── tests/
│
├── config.yaml
├── .env
├── requirements.txt
└── Dockerfile
```

### `schemas.py`

```python
from pydantic import BaseModel, Field


class PredictionRequest(BaseModel):

    age: int = Field(gt=0)

    income: float = Field(gt=0)

    tenure: int = Field(ge=0)


class PredictionResponse(BaseModel):

    prediction: int

    model_version: str

    probability: float = Field(
        ge=0,
        le=1
    )
```

### `config.py`

```python
from pydantic_settings import BaseSettings


class Settings(BaseSettings):

    model_name: str
    model_version: str
    model_path: str
    mlflow_uri: str

    class Config:
        env_file = ".env"


settings = Settings()
```

### `main.py`

```python
from fastapi import FastAPI
from schemas import PredictionRequest, PredictionResponse

app = FastAPI()


@app.post(
    "/predict",
    response_model=PredictionResponse
)
def predict(request: PredictionRequest):

    # model prediction
    prediction = 1
    probability = 0.92

    return PredictionResponse(
        prediction=prediction,
        model_version="1.0",
        probability=probability
    )
```

Now Pydantic is controlling both sides:

```text
                  MLOps System

             ┌──────────────┐
             │ Configuration│
             └──────┬───────┘
                    │
                    ↓
               Pydantic
                    │
          ┌─────────┴─────────┐
          ↓                   ↓
    Training Config       API Config
          │                   │
          ↓                   ↓
       Training          FastAPI
          │                   │
          ↓                   ↓
        Model             Prediction
          │                   │
          └─────────┬─────────┘
                    ↓
               Pydantic
                    ↓
              Output Contract
```

---

# 17. Pydantic vs Pandas

This distinction is important.

### Pydantic

Best for:

```text
API request
API response
Configuration
Parameters
Metadata
Data contracts
```

### Pandas

Best for:

```text
DataFrame
Data cleaning
Data transformation
Data analysis
Feature engineering
```

For example:

```text
Raw JSON
   ↓
Pydantic
   ↓
Validated data
   ↓
Pandas
   ↓
Feature engineering
   ↓
ML model
```

---

# 18. Pydantic vs Dataclass

You may also encounter:

```python
from dataclasses import dataclass
```

Dataclass:

```python
@dataclass
class TrainingConfig:
    epochs: int
    learning_rate: float
```

Pydantic:

```python
class TrainingConfig(BaseModel):
    epochs: int
    learning_rate: float
```

The key difference:

**Dataclass primarily provides convenient structured Python objects.**

**Pydantic additionally provides runtime validation, parsing, serialization, and schema generation.**

For ML APIs and production configuration, Pydantic is particularly useful.

---

# 19. What you should learn for MLOps

Since you're learning MLOps, I would learn Pydantic in this order:

### Level 1 — Basics

Learn:

```text
BaseModel
Field
Type hints
Validation
Optional fields
Default values
Nested models
```

### Level 2 — Production

Learn:

```text
model_dump()
model_validate()
ValidationError
Custom validators
JSON schema
Serialization
```

### Level 3 — MLOps

Learn:

```text
Pydantic + YAML
Pydantic + .env
Pydantic + FastAPI
Pydantic + MLflow
Pydantic + Optuna
Pydantic + Docker
Pydantic + CI/CD
```

### Level 4 — Production architecture

Eventually build:

```text
Git
 │
 ↓
CI/CD
 │
 ↓
Config validation ← Pydantic
 │
 ↓
Data validation
 │
 ↓
Training
 │
 ├── Optuna
 │
 └── MLflow
 │
 ↓
Model Registry
 │
 ↓
Docker
 │
 ↓
FastAPI
 │
 ↓
Pydantic request validation
 │
 ↓
Model
 │
 ↓
Pydantic response validation
```

**The main idea to remember:** Pydantic is not an ML library. It is a **validation and data-contract layer** that makes ML/MLOps systems safer and more predictable.

If you're building your MLOps skills step-by-step, the next practical exercise should be **“Pydantic + YAML + FastAPI + XGBoost”**: create one small project where Pydantic validates the training configuration, hyperparameters, API request, and API response.
